# Ajuste fino do leitor (QLoRA) — passos 5 a 7

IC0024 (PGCOMP/UFBA). Este notebook gera pares de treino a partir de um **documento público** (um regulamento
portuário publicado por uma autoridade portuária), ajusta o `Qwen3-4B` com QLoRA (Unsloth) e exporta o leitor
ajustado em GGUF `q4_k_m` para ser registrado no Ollama local como `qwen3-manual:4b`.

**Autossuficiente.** O código do projeto que este notebook usa vem embutido na seção 3; nada é baixado do
repositório. Da internet vêm só pacotes Python, o Ollama, o modelo `llama3.2:3b` e o `Qwen3-4B`.

> **Confidencialidade.** Este notebook roda na nuvem do Google. **Nunca** envie para cá o manual da empresa
> (`data/manual.pdf`), nem `data/trechos.jsonl`, nem qualquer arquivo derivado dele. Só o documento público entra
> aqui. Do Colab voltam apenas o GGUF, `treino_log.csv` e `metricas_validacao.json`.

**Antes de começar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois,
*Ambiente de execução → Executar tudo*; na seção 4, mude `DOCUMENTO_E_PUBLICO` para `True` e envie o PDF.

| Passo | O que faz | Saída |
|---|---|---|
| 1–3 | ingestão e trechos do documento público, com o mesmo código do projeto | `trechos_publico.jsonl` |
| 5 | `llama3.2:3b` gera pergunta + resposta; prompt do leitor com 3 a 5 trechos | `treino.jsonl`, `validacao.jsonl` |
| 6 | QLoRA (Unsloth) sobre `unsloth/Qwen3-4B` em 4 bits, perda só na resposta | `adapter/`, `treino_log.csv` |
| 7 | merge + GGUF `q4_k_m` | `qwen3-manual-q4_k_m.gguf` |

## 1. Parâmetros

In [ ]:
N_MAX_TRECHOS = 1500  # trechos do documento público usados para gerar exemplos
FRAC_SEM_RESPOSTA = 0.20  # exemplos cujo alvo é exatamente "Não encontrado no manual."
FRAC_VALIDACAO = 0.05
MAX_TOKENS_CONTEXTO = 1400  # estimativa (1,3 token/palavra) dos trechos no prompt, para caber em 2048 tokens
SEED = 42

## 2. Instalação (Unsloth, Ollama e dependências do código do projeto)

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q trl datasets
!pip install -q pymupdf bm25s nltk ollama pyyaml tqdm
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

servidor_ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("o Ollama não subiu no Colab")
!ollama pull llama3.2:3b

## 3. Código do projeto (embutido)

As células abaixo gravam em `/content/qa_manual_colab/` os arquivos do projeto que o Colab usa: os módulos de `qa_manual`
dos passos 1 a 3 e 5, o `configs/base.yaml` e os prompts. Elas foram geradas por `colab/gerar_notebook.py` a
partir do repositório; não edite aqui, edite o código local e gere o notebook de novo.

In [ ]:
import os
import sys

PASTA_CODIGO = "/content/qa_manual_colab"
for sub in ("qa_manual", "configs", "prompts"):
    os.makedirs(os.path.join(PASTA_CODIGO, sub), exist_ok=True)

In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/__init__.py
"""Sistema de QA sobre manual técnico com Retriever-Reader local (Ollama). IC0024, PGCOMP/UFBA."""

__version__ = "0.2.0"


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/io_utils.py
"""Entrada e saída: JSONL/JSON, hashes, carimbos de tempo, run_id e log de execução.

Regra de confidencialidade: logs registram ids, contagens, tempos e hashes; nunca texto do manual.
"""

from __future__ import annotations

import hashlib
import json
import logging
import subprocess
from collections.abc import Iterable, Iterator
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RAIZ = Path(__file__).resolve().parent.parent
FORMATO_LOG = "%(asctime)s %(levelname)s %(name)s: %(message)s"


class ErroUsuario(Exception):
    """Erro de entrada ou de ambiente com mensagem que diz o que fazer.

    A CLI imprime a mensagem e encerra com ``codigo`` (2 para entradas inválidas).
    """

    codigo = 2


def pasta_sincronizada(caminho: str | Path) -> str | None:
    """Nome do serviço de nuvem que sincroniza ``caminho``, ou ``None`` se a pasta é só local.

    Detecta iCloud Drive (inclusive Mesa e Documentos sincronizados no macOS), pastas em
    ``~/Library/CloudStorage`` (Dropbox, OneDrive, Google Drive) e as pastas clássicas desses serviços. No
    iCloud, um componente terminado em ``.nosync`` exclui a pasta da sincronização.
    """
    p = Path(caminho).expanduser().resolve()
    casa = Path.home().resolve()
    icloud = casa / "Library" / "Mobile Documents"
    servico = None
    if p.is_relative_to(icloud):
        servico = "iCloud Drive"
    for pasta in ("Desktop", "Documents"):
        if (icloud / "com~apple~CloudDocs" / pasta).is_symlink() and p.is_relative_to(casa / pasta):
            servico = f"iCloud Drive (pasta {pasta} sincronizada)"
    if servico and any(parte.endswith(".nosync") for parte in p.parts):
        servico = None
    nuvens = casa / "Library" / "CloudStorage"
    if p.is_relative_to(nuvens) and p != nuvens:
        servico = p.relative_to(nuvens).parts[0]
    for nome in ("Dropbox", "OneDrive", "Google Drive"):
        if p.is_relative_to(casa / nome):
            servico = nome
    return servico


# ---------------------------------------------------------------------------
# Arquivos
# ---------------------------------------------------------------------------


def validar_arquivo(caminho: str | Path, extensoes: tuple[str, ...] = ()) -> Path:
    """Confere se o arquivo existe e tem uma das extensões esperadas.

    Args:
        caminho: caminho informado pelo usuário.
        extensoes: extensões aceitas (com ponto, minúsculas); vazio aceita qualquer uma.

    Returns:
        O caminho como ``Path``.

    Raises:
        ErroUsuario: se o arquivo não existe, não é arquivo ou tem extensão inesperada.
    """
    p = Path(caminho).expanduser()
    if not p.exists():
        raise ErroUsuario(f"Arquivo não encontrado: {p}")
    if not p.is_file():
        raise ErroUsuario(f"O caminho não é um arquivo: {p}")
    if extensoes and p.suffix.lower() not in extensoes:
        raise ErroUsuario(f"Extensão inválida para {p.name}: esperado {', '.join(extensoes)}.")
    return p


def ler_jsonl(caminho: str | Path) -> list[dict]:
    """Lê um arquivo JSONL (uma linha por registro), ignorando linhas vazias.

    Raises:
        ErroUsuario: se alguma linha não for um objeto JSON válido (a mensagem cita só o número da linha).
    """
    itens = []
    with open(caminho, encoding="utf-8") as f:
        for n, linha in enumerate(f, 1):
            linha = linha.strip()
            if not linha:
                continue
            try:
                item = json.loads(linha)
            except json.JSONDecodeError as e:
                raise ErroUsuario(f"{caminho}, linha {n}: JSON inválido ({e.msg}).") from None
            if not isinstance(item, dict):
                raise ErroUsuario(f"{caminho}, linha {n}: esperado um objeto JSON.")
            itens.append(item)
    return itens


def escrever_jsonl(caminho: str | Path, itens: Iterable[dict]) -> None:
    """Grava os registros em JSONL (UTF-8), criando o diretório se preciso."""
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, "w", encoding="utf-8") as f:
        for item in itens:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def anexar_jsonl(caminho: str | Path, item: dict) -> None:
    """Acrescenta um registro e força a escrita, para sobreviver a interrupções."""
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()


def ler_json(caminho: str | Path) -> Any:
    """Lê um arquivo JSON."""
    with open(caminho, encoding="utf-8") as f:
        return json.load(f)


def escrever_json(caminho: str | Path, dados: Any) -> None:
    """Grava ``dados`` como JSON indentado (UTF-8)."""
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2, default=str)
        f.write("\n")


def sha256_arquivo(caminho: str | Path) -> str:
    """Hash SHA-256 do conteúdo do arquivo."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


def sha256_texto(texto: str) -> str:
    """Hash SHA-256 de um texto (UTF-8)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def lotes(itens: list, tamanho: int) -> Iterator[list]:
    """Divide ``itens`` em lotes consecutivos de até ``tamanho`` elementos."""
    for i in range(0, len(itens), tamanho):
        yield itens[i : i + tamanho]


# ---------------------------------------------------------------------------
# Tempo, execuções e versão
# ---------------------------------------------------------------------------


def agora_iso() -> str:
    """Instante atual em UTC no formato ``2026-10-06T14:30:00Z``."""
    return datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")


def carimbo() -> str:
    """Carimbo local usado em nomes de execução: ``2026-10-10T18-02-11``."""
    return datetime.now().strftime("%Y-%m-%dT%H-%M-%S")


def novo_run_id(*partes: str) -> str:
    """Cria um ``run_id`` com timestamp, por exemplo ``2026-10-10T18-02-11_S3_ajustado``."""
    return "_".join([carimbo(), *[p for p in partes if p]])


def criar_run_dir(runs_dir: str | Path, run_id: str) -> Path:
    """Cria (se preciso) e devolve ``runs_dir/run_id``."""
    caminho = Path(runs_dir) / run_id
    caminho.mkdir(parents=True, exist_ok=True)
    return caminho


def versao_codigo() -> str:
    """Hash curto do commit atual (``git rev-parse --short HEAD``) ou ``'sem-git'``."""
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=RAIZ, capture_output=True, text=True, timeout=5)
    except (OSError, subprocess.SubprocessError):
        return "sem-git"
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else "sem-git"


def anexar_log_arquivo(caminho: str | Path) -> logging.Handler:
    """Passa a gravar o log do pacote ``qa_manual`` também em ``caminho`` (por exemplo ``log.txt``).

    Returns:
        O handler criado, para ser removido com :func:`remover_log_arquivo` ao fim da execução.
    """
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    handler = logging.FileHandler(caminho, encoding="utf-8")
    handler.setFormatter(logging.Formatter(FORMATO_LOG))
    handler.setLevel(logging.INFO)
    logger = logging.getLogger("qa_manual")
    logger.addHandler(handler)
    if logger.level == logging.NOTSET or logger.level > logging.INFO:
        logger.setLevel(logging.INFO)
    return handler


def remover_log_arquivo(handler: logging.Handler) -> None:
    """Desliga e fecha um handler criado por :func:`anexar_log_arquivo`."""
    logging.getLogger("qa_manual").removeHandler(handler)
    handler.close()


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/config.py
"""Carregamento e validação da configuração YAML (``configs/``) em dataclasses imutáveis.

``carregar(base, *sobrescritas)`` lê ``base.yaml`` e aplica as sobrescritas em ordem. Seções aninhadas são
mescladas chave a chave, de modo que um YAML pode alterar só ``trechos.max_tokens`` sem repetir a seção
inteira. Chaves desconhecidas, tipos errados e valores fora do domínio encerram com :class:`ErroUsuario`.
"""

from __future__ import annotations

import dataclasses
import os
import re
import types
import typing
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import yaml

from .io_utils import RAIZ, ErroUsuario

DIR_CONFIGS = RAIZ / "configs"
BASE_PADRAO = "configs/base.yaml"
MODOS_RETRIEVER = ("nenhum", "bm25", "denso", "hibrido")
LEITORES = ("base", "ajustado")


@dataclass(frozen=True)
class Paths:
    manual_pdf: Path
    trechos: Path
    index_dir: Path
    runs_dir: Path
    gold_dev: Path
    gold_test: Path
    ppl_holdout: Path


@dataclass(frozen=True)
class OllamaCfg:
    host: str
    timeout_s: float


@dataclass(frozen=True)
class Modelos:
    leitor_base: str
    leitor_ajustado: str
    embeddings: str
    gerador: str


@dataclass(frozen=True)
class Ingestao:
    remover_cabecalhos_rodapes: bool
    limiar_repeticao_cabecalho: float
    regex_titulo: str
    validar_sequencia_titulos: bool
    min_chars_paragrafo: int


@dataclass(frozen=True)
class TrechosCfg:
    max_tokens: int
    min_tokens: int
    tokens_por_palavra: float
    sobreposicao_paragrafos: int


@dataclass(frozen=True)
class BM25Cfg:
    k1: float
    b: float
    remover_stopwords: bool
    manter_acentos: bool


@dataclass(frozen=True)
class DensoCfg:
    dimensao: int
    normalizar_l2: bool
    batch_size: int


@dataclass(frozen=True)
class Decodificacao:
    temperature: float
    top_p: float
    seed: int
    num_ctx: int
    num_predict: int
    think: bool

    def opcoes(self) -> dict:
        """Opções de amostragem no formato do Ollama."""
        return {
            "temperature": self.temperature,
            "top_p": self.top_p,
            "seed": self.seed,
            "num_ctx": self.num_ctx,
            "num_predict": self.num_predict,
        }


@dataclass(frozen=True)
class Abstencao:
    frase: str
    limiar_score: float | None


@dataclass(frozen=True)
class GoldCfg:
    n_candidatas: int
    frac_sem_resposta: float
    frac_dev: float
    seed: int


@dataclass(frozen=True)
class Avaliacao:
    configs: list[str]
    leitores: list[str]
    repeticoes: int
    bertscore_modelo: str
    bertscore_num_layers: int
    ppl_ctx: int


@dataclass(frozen=True)
class Config:
    """Configuração efetiva de uma execução. ``nome`` identifica a configuração experimental (S0 a S3)."""

    paths: Paths
    ollama: OllamaCfg
    modelos: Modelos
    ingestao: Ingestao
    trechos: TrechosCfg
    bm25: BM25Cfg
    denso: DensoCfg
    retriever: str
    k: int
    k_candidatos_por_lista: int
    rrf_c: int
    decodificacao: Decodificacao
    abstencao: Abstencao
    gold: GoldCfg
    avaliacao: Avaliacao
    nome: str = "base"

    def modelo_leitor(self, leitor: str) -> str:
        """Nome do modelo no Ollama para o leitor ``base`` ou ``ajustado``."""
        if leitor == "base":
            return self.modelos.leitor_base
        if leitor == "ajustado":
            return self.modelos.leitor_ajustado
        raise ErroUsuario(f"Leitor desconhecido: {leitor}. Use: {', '.join(LEITORES)}.")

    def como_dict(self) -> dict:
        """Configuração completa como dicionário serializável (caminhos viram texto)."""
        return _serializavel(dataclasses.asdict(self))

    def com(self, **mudancas: Any) -> Config:
        """Cópia com campos de primeiro nível alterados, por exemplo ``cfg.com(k=8)``."""
        return dataclasses.replace(self, **mudancas)


# Campos preenchidos pelo carregador, não pelo YAML.
_CAMPOS_INTERNOS = {"nome"}


def _serializavel(valor: Any) -> Any:
    if isinstance(valor, dict):
        return {k: _serializavel(v) for k, v in valor.items()}
    if isinstance(valor, list | tuple):
        return [_serializavel(v) for v in valor]
    if isinstance(valor, Path):
        return str(valor)
    return valor


def resolver_config(ref: str | Path) -> Path:
    """Localiza um YAML de configuração.

    Aceita um nome curto (``S3`` vira ``configs/S3.yaml``) ou um caminho. Caminhos relativos são procurados
    primeiro no diretório atual e depois na raiz do repositório.
    """
    ref = Path(ref)
    candidatos = [DIR_CONFIGS / f"{ref}.yaml"] if not ref.suffix else [ref, RAIZ / ref]
    for c in candidatos:
        if c.is_file():
            return c
    disponiveis = ", ".join(sorted(p.stem for p in DIR_CONFIGS.glob("*.yaml")))
    raise ErroUsuario(f"Configuração não encontrada: {ref}. Disponíveis em configs/: {disponiveis}.")


def _ler_yaml(caminho: Path) -> dict:
    try:
        dados = yaml.safe_load(caminho.read_text(encoding="utf-8")) or {}
    except yaml.YAMLError as e:
        raise ErroUsuario(f"YAML inválido em {caminho}: {e}") from None
    if not isinstance(dados, dict):
        raise ErroUsuario(f"{caminho}: o YAML deve ser um mapeamento de chaves.")
    return dados


def mesclar(base: dict, sobrescrita: dict) -> dict:
    """Mescla ``sobrescrita`` sobre ``base``; seções (dicionários) são mescladas recursivamente."""
    saida = dict(base)
    for chave, valor in sobrescrita.items():
        if isinstance(valor, dict) and isinstance(saida.get(chave), dict):
            saida[chave] = mesclar(saida[chave], valor)
        else:
            saida[chave] = valor
    return saida


def _converter(valor: Any, tipo: Any, caminho: str) -> Any:
    """Valida ``valor`` contra a anotação ``tipo`` e converte quando necessário."""
    origem = typing.get_origin(tipo)
    if origem in (typing.Union, types.UnionType):
        opcoes = typing.get_args(tipo)
        if valor is None and type(None) in opcoes:
            return None
        tipo = next(t for t in opcoes if t is not type(None))
        origem = typing.get_origin(tipo)
    if dataclasses.is_dataclass(tipo):
        if not isinstance(valor, dict):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser uma seção (mapeamento).")
        return _construir(tipo, valor, caminho)
    if origem is list:
        (tipo_item,) = typing.get_args(tipo)
        if not isinstance(valor, list):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser uma lista.")
        return [_converter(v, tipo_item, f"{caminho}[{i}]") for i, v in enumerate(valor)]
    if tipo is Path:
        if not isinstance(valor, str) or not valor:
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser um caminho (texto).")
        return Path(valor)
    if tipo is bool:
        if not isinstance(valor, bool):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser true ou false.")
        return valor
    if tipo is int:
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser um número inteiro.")
        return valor
    if tipo is float:
        if isinstance(valor, bool) or not isinstance(valor, int | float):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser um número.")
        return float(valor)
    if tipo is str:
        if not isinstance(valor, str):
            raise ErroUsuario(f"Configuração: '{caminho}' deve ser texto.")
        return valor
    raise TypeError(f"Tipo de configuração não suportado: {tipo}")  # pragma: no cover


def _construir(cls: type, dados: dict, prefixo: str = "") -> Any:
    dicas = typing.get_type_hints(cls)
    campos = [f for f in dataclasses.fields(cls) if f.name not in _CAMPOS_INTERNOS]
    nomes = {f.name for f in campos}
    desconhecidas = sorted(set(dados) - nomes)
    if desconhecidas:
        lista = ", ".join(f"{prefixo}{k}" for k in desconhecidas)
        raise ErroUsuario(f"Configuração: chave(s) desconhecida(s): {lista}.")
    faltando = sorted(n for n in nomes if n not in dados)
    if faltando:
        lista = ", ".join(f"{prefixo}{k}" for k in faltando)
        raise ErroUsuario(f"Configuração: chave(s) obrigatória(s) ausente(s): {lista}.")
    valores = {f.name: _converter(dados[f.name], dicas[f.name], f"{prefixo}{f.name}") for f in campos}
    return cls(**valores)


def _validar_dominio(cfg: Config) -> None:
    erros = []
    if cfg.retriever not in MODOS_RETRIEVER:
        erros.append(f"retriever deve ser um de {', '.join(MODOS_RETRIEVER)}")
    if cfg.k < 1:
        erros.append("k deve ser >= 1")
    if cfg.k_candidatos_por_lista < cfg.k:
        erros.append("k_candidatos_por_lista deve ser >= k")
    if cfg.rrf_c < 0:
        erros.append("rrf_c deve ser >= 0")
    if not 0 < cfg.ingestao.limiar_repeticao_cabecalho <= 1:
        erros.append("ingestao.limiar_repeticao_cabecalho deve estar em (0, 1]")
    try:
        re.compile(cfg.ingestao.regex_titulo)
    except re.error as e:
        erros.append(f"ingestao.regex_titulo inválida ({e})")
    t = cfg.trechos
    if t.max_tokens < 1 or t.min_tokens < 0 or t.min_tokens > t.max_tokens:
        erros.append("trechos: exige 0 <= min_tokens <= max_tokens e max_tokens >= 1")
    if t.tokens_por_palavra <= 0:
        erros.append("trechos.tokens_por_palavra deve ser > 0")
    if t.sobreposicao_paragrafos < 0:
        erros.append("trechos.sobreposicao_paragrafos deve ser >= 0")
    if cfg.denso.dimensao < 1 or cfg.denso.batch_size < 1:
        erros.append("denso.dimensao e denso.batch_size devem ser >= 1")
    if not 0 <= cfg.gold.frac_sem_resposta < 1 or not 0 < cfg.gold.frac_dev < 1:
        erros.append("gold.frac_sem_resposta deve estar em [0, 1) e gold.frac_dev em (0, 1)")
    if cfg.gold.n_candidatas < 1:
        erros.append("gold.n_candidatas deve ser >= 1")
    invalidos = [lt for lt in cfg.avaliacao.leitores if lt not in LEITORES]
    if invalidos:
        erros.append(f"avaliacao.leitores aceita apenas {', '.join(LEITORES)}")
    if cfg.avaliacao.repeticoes < 1:
        erros.append("avaliacao.repeticoes deve ser >= 1")
    if not cfg.abstencao.frase.strip():
        erros.append("abstencao.frase não pode ser vazia")
    if erros:
        raise ErroUsuario("Configuração inválida:\n  - " + "\n  - ".join(erros))


def carregar(base: str | Path = BASE_PADRAO, *sobrescritas: str | Path | None, nome: str | None = None) -> Config:
    """Carrega ``base`` e aplica as sobrescritas em ordem.

    Args:
        base: caminho do YAML base (padrão ``configs/base.yaml``).
        *sobrescritas: YAMLs ou nomes curtos (``S3``) aplicados em sequência; ``None`` é ignorado.
        nome: nome da configuração experimental; por padrão, o nome do último YAML de sobrescrita ou ``base``.

    Returns:
        A configuração validada. A variável de ambiente ``OLLAMA_HOST``, se definida, substitui
        ``ollama.host`` (o cliente continua aceitando só endereços locais).

    Raises:
        ErroUsuario: se algum arquivo não existir ou a configuração for inválida.
    """
    dados = _ler_yaml(resolver_config(base))
    ultimo = None
    for s in sobrescritas:
        if s is None:
            continue
        caminho = resolver_config(s)
        dados = mesclar(dados, _ler_yaml(caminho))
        ultimo = caminho.stem
    host = os.environ.get("OLLAMA_HOST")
    if host:
        dados = mesclar(dados, {"ollama": {"host": host if "://" in host else f"http://{host}"}})
    cfg = _construir(Config, dados)
    cfg = dataclasses.replace(cfg, nome=nome or ultimo or "base")
    _validar_dominio(cfg)
    return cfg


def carregar_experimento(exp: str, base: str | Path = BASE_PADRAO, sobrescrever: str | Path | None = None) -> Config:
    """Configuração de um experimento (``S0`` a ``S3``).

    Ordem de aplicação: base, depois ``sobrescrever``, depois ``configs/<exp>.yaml``.
    """
    return carregar(base, sobrescrever, exp, nome=Path(exp).stem)


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/ollama_client.py
"""Cliente fino sobre o Ollama (embeddings e chat), com uma implementação falsa para testes.

Confidencialidade: :class:`OllamaReal` só aceita servidores locais (localhost, 127.0.0.1, ::1). Nenhum texto do
manual pode ser enviado a outro endereço.
"""

from __future__ import annotations

import hashlib
import logging
import re
import unicodedata
from collections.abc import Callable
from typing import TYPE_CHECKING, Protocol
from urllib.parse import urlparse

from .io_utils import ErroUsuario

if TYPE_CHECKING:
    from .config import Config

log = logging.getLogger(__name__)

HOSTS_LOCAIS = {"localhost", "127.0.0.1", "::1", "0.0.0.0"}
MODELOS_CRIADOS_LOCALMENTE = {"qwen3-manual:4b"}


class OllamaIndisponivel(ErroUsuario):
    """O servidor do Ollama não respondeu."""

    codigo = 1


class ModeloAusente(ErroUsuario):
    """O modelo pedido não está instalado no Ollama."""

    codigo = 1


def instrucao_modelo(modelo: str) -> str:
    """Mensagem que diz como obter um modelo ausente."""
    if modelo in MODELOS_CRIADOS_LOCALMENTE or modelo.endswith("-manual:4b"):
        return (
            f"modelo {modelo} ausente; crie com `ollama create {modelo} -f Modelfile` "
            "(ver docs/README.md, seção Ajuste fino no Colab)"
        )
    return f"modelo {modelo} ausente; rode `ollama pull {modelo}`"


def modelo_presente(modelo: str, disponiveis: list[str]) -> bool:
    """Compara nomes do Ollama tolerando o sufixo implícito ``:latest``."""
    alvo = modelo if ":" in modelo else f"{modelo}:latest"
    return any(d == modelo or d == alvo for d in disponiveis)


def validar_host_local(host: str) -> str:
    """Garante que ``host`` aponta para a própria máquina e devolve a URL sem barra final."""
    if "://" not in host:
        host = f"http://{host}"
    nome = urlparse(host).hostname or ""
    if nome not in HOSTS_LOCAIS:
        raise ErroUsuario(
            f"O endereço do Ollama não é local ({host}). O manual é confidencial e não pode sair da máquina."
        )
    return host.rstrip("/")


class OllamaClient(Protocol):
    """Interface usada pelo restante do código (permite injetar :class:`OllamaFake` nos testes)."""

    def embed(self, model: str, input: list[str]) -> list[list[float]]:
        """Embeddings de uma lista de textos."""
        ...

    def chat(
        self,
        model: str,
        messages: list[dict],
        options: dict,
        think: bool | None = None,
        format: str | None = None,
    ) -> dict:
        """Uma rodada de chat; devolve ``{"content", "model", "prompt_eval_count", "eval_count"}``."""
        ...

    def modelos_disponiveis(self) -> list[str]:
        """Nomes dos modelos instalados."""
        ...


class OllamaReal:
    """Implementação sobre o pacote ``ollama`` (``ollama.Client(host=...)``).

    ``chat`` devolve ``{"content", "model", "prompt_eval_count", "eval_count"}``.
    """

    def __init__(self, host: str = "http://localhost:11434", timeout_s: float = 300):
        import ollama

        self.host = validar_host_local(host)
        self.timeout_s = timeout_s
        self._ollama = ollama
        self._cliente = ollama.Client(host=self.host, timeout=timeout_s)
        self._sem_think: set[str] = set()

    def _chamar(self, fn: Callable, modelo: str | None = None):
        import httpx

        try:
            return fn()
        except self._ollama.ResponseError as e:
            if e.status_code == 404 and modelo:
                raise ModeloAusente(instrucao_modelo(modelo)) from None
            raise
        except (ConnectionError, httpx.ConnectError):
            raise OllamaIndisponivel(f"Ollama não respondeu em {self.host}; rode `ollama serve`") from None
        except httpx.TimeoutException:
            raise OllamaIndisponivel(
                f"Ollama em {self.host} não respondeu em {self.timeout_s:.0f} s; aumente ollama.timeout_s"
            ) from None

    def embed(self, model: str, input: list[str]) -> list[list[float]]:
        """Embeddings de uma lista de textos."""
        r = self._chamar(lambda: self._cliente.embed(model=model, input=list(input)), model)
        vetores = [list(v) for v in r.embeddings]
        if len(vetores) != len(input):
            raise RuntimeError(f"/api/embed devolveu {len(vetores)} vetores para {len(input)} textos.")
        return vetores

    def chat(
        self,
        model: str,
        messages: list[dict],
        options: dict,
        think: bool | None = None,
        format: str | None = None,
    ) -> dict:
        """Uma rodada de chat sem streaming."""
        kwargs: dict = {"model": model, "messages": messages, "options": options, "stream": False}
        if format:
            kwargs["format"] = format
        if think is not None and model not in self._sem_think:
            kwargs["think"] = think
        try:
            r = self._chamar(lambda: self._cliente.chat(**kwargs), model)
        except self._ollama.ResponseError as e:
            # Modelos sem suporte a raciocínio (ex.: llama3.2) rejeitam o parâmetro think.
            if "think" in kwargs and "think" in str(e).lower():
                self._sem_think.add(model)
                kwargs.pop("think")
                r = self._chamar(lambda: self._cliente.chat(**kwargs), model)
            else:
                raise
        return {
            "content": r.message.content or "",
            "model": r.model or model,
            "prompt_eval_count": r.prompt_eval_count,
            "eval_count": r.eval_count,
        }

    def modelos_disponiveis(self) -> list[str]:
        """Nomes dos modelos instalados (``ollama list``)."""
        r = self._chamar(lambda: self._cliente.list())
        return [m.model for m in r.models if m.model]

    def versao(self) -> str:
        """Versão do servidor (``/api/version``) ou ``'?'``."""
        import httpx

        try:
            return httpx.get(f"{self.host}/api/version", timeout=10).json().get("version", "?")
        except (httpx.HTTPError, ValueError):
            return "?"


def criar_cliente(cfg: Config) -> OllamaReal:
    """Cliente real a partir de ``cfg.ollama``."""
    return OllamaReal(cfg.ollama.host, cfg.ollama.timeout_s)


# ---------------------------------------------------------------------------
# Implementação falsa (testes)
# ---------------------------------------------------------------------------

_TOKEN = re.compile(r"[^\W_]{2,}")


def vetor_hash(texto: str, dimensao: int) -> list[float]:
    """Embedding determinístico: contagem de tokens espalhada por hash (com semântica léxica)."""
    v = [0.0] * dimensao
    for tok in _TOKEN.findall(unicodedata.normalize("NFC", texto.lower())):
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dimensao] += 1.0
    if not any(v):  # texto sem tokens: um componente fixo derivado do hash do texto
        v[int(hashlib.md5(texto.encode()).hexdigest(), 16) % dimensao] = 1.0
    return v


class OllamaFake:
    """Cliente falso: embeddings por hash e chat com texto fixo ou calculado por uma função.

    Args:
        dimensao: dimensão dos vetores devolvidos por ``embed``.
        resposta: texto fixo ou função ``(modelo, prompt) -> texto`` para o chat.
        modelos: nomes devolvidos por ``modelos_disponiveis``.
    """

    def __init__(
        self,
        dimensao: int = 1024,
        resposta: str | Callable[[str, str], str] = "Não encontrado no manual.",
        modelos: list[str] | None = None,
    ):
        self.host = "fake://ollama"
        self.dimensao = dimensao
        self.resposta = resposta
        self.modelos = (
            modelos
            if modelos is not None
            else [
                "qwen3:4b",
                "qwen3-manual:4b",
                "bge-m3:latest",
                "llama3.2:3b",
            ]
        )
        self.chamadas_chat: list[dict] = []
        self.n_textos_embed = 0

    def embed(self, model: str, input: list[str]) -> list[list[float]]:
        """Vetores de :func:`vetor_hash`."""
        self.n_textos_embed += len(input)
        return [vetor_hash(t, self.dimensao) for t in input]

    def chat(
        self,
        model: str,
        messages: list[dict],
        options: dict,
        think: bool | None = None,
        format: str | None = None,
    ) -> dict:
        """Registra a chamada e devolve o texto fixo ou o da função ``resposta``."""
        prompt = messages[-1]["content"]
        self.chamadas_chat.append(
            {"model": model, "prompt": prompt, "options": dict(options), "think": think, "format": format}
        )
        texto = self.resposta(model, prompt) if callable(self.resposta) else self.resposta
        return {
            "content": texto,
            "model": model,
            "prompt_eval_count": len(prompt.split()),
            "eval_count": len(texto.split()),
        }

    def modelos_disponiveis(self) -> list[str]:
        """Modelos configurados no construtor."""
        return list(self.modelos)

    def versao(self) -> str:
        """Versão fictícia."""
        return "fake"


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/ingest.py
"""Passos 1 e 2: texto do PDF por página (PyMuPDF) e parágrafos com tema e página.

Fluxo: blocos de texto (``page.get_text("blocks")``) -> remoção de cabeçalhos/rodapés repetidos -> limpeza
(hifenização, quebras de linha) -> detecção de títulos pela regex configurada -> propagação do tema corrente ->
descarte de blocos curtos. Títulos viram parágrafos com ``eh_titulo=True``: só definem o tema e não entram no
texto dos trechos.
"""

from __future__ import annotations

import logging
import re
import unicodedata
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import TYPE_CHECKING

import pymupdf

from .io_utils import ErroUsuario

if TYPE_CHECKING:
    from .config import Config

log = logging.getLogger(__name__)

TEMA_INICIAL = "Sem título"
MIN_CHARS_PAGINA = 50  # página com menos caracteres que isso conta como "sem texto"
FRAC_PAGINAS_SEM_TEXTO = 0.8  # acima dessa fração de páginas sem texto, o PDF precisa de OCR
MAX_SALTO_NUMERACAO = 5  # maior salto aceito entre seções irmãs (2.3 -> 2.8)
# Entrada de sumário: "4.2 Liberação de carga ........ 17". Linhas assim não são títulos nem conteúdo.
_SUMARIO = re.compile(r"(\.\s?){4,}\s*\d+\s*$|…+\s*\d+\s*$")
_CONTROLE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
# Só ligaduras tipográficas: NFKC também trocaria "nº" por "no" e "m³" por "m3".
_LIGADURAS = str.maketrans({"ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st"})


@dataclass
class Paragrafo:
    """Parágrafo extraído do PDF, já associado ao tema (seção) corrente."""

    idx: int
    pagina: int
    tema: str
    tema_id: str
    texto: str
    eh_titulo: bool


def _abrir(pdf_path: Path) -> pymupdf.Document:
    try:
        doc = pymupdf.open(pdf_path)
    except Exception as e:
        raise ErroUsuario(f"Não foi possível abrir o PDF {pdf_path}: {type(e).__name__}.") from None
    if not doc.is_pdf:
        doc.close()
        raise ErroUsuario(f"{pdf_path} não é um PDF.")
    if doc.needs_pass:
        doc.close()
        raise ErroUsuario(f"{pdf_path} está protegido por senha; remova a proteção antes de indexar.")
    return doc


def _extrair(pdf_path: Path) -> tuple[list[tuple[int, str]], int]:
    blocos: list[tuple[int, str]] = []
    with _abrir(pdf_path) as doc:
        n_paginas = doc.page_count
        for num, pagina in enumerate(doc, start=1):
            # (x0, y0, x1, y1, texto, numero_bloco, tipo); tipo 1 = imagem
            texto = [b for b in pagina.get_text("blocks") if b[6] == 0 and b[4].strip()]
            texto.sort(key=lambda b: (round(b[1], 1), b[0]))
            blocos.extend((num, b[4]) for b in texto)
    return blocos, n_paginas


def extrair_blocos(pdf_path: Path) -> list[tuple[int, str]]:
    """Blocos de texto de todas as páginas, na ordem de leitura (y, depois x).

    Args:
        pdf_path: caminho do PDF.

    Returns:
        Lista de ``(pagina, texto_do_bloco)`` com página 1-based. Blocos de imagem são ignorados.
    """
    return _extrair(Path(pdf_path))[0]


def _chave_repeticao(texto: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"\d", "", texto.strip())).strip()


def detectar_cabecalhos_rodapes(blocos: list[tuple[int, str]], n_paginas: int, limiar: float) -> set[str]:
    """Identifica cabeçalhos e rodapés pela repetição entre páginas.

    Cada bloco é normalizado (strip, remoção de dígitos, espaços colapsados) e contado uma vez por página. Os
    que aparecem em mais de ``limiar * n_paginas`` páginas (e em pelo menos duas) são cabeçalho ou rodapé.

    Returns:
        Conjunto de chaves normalizadas a remover (use a mesma normalização para filtrar).
    """
    paginas_por_chave: dict[str, set[int]] = defaultdict(set)
    for pagina, texto in blocos:
        paginas_por_chave[_chave_repeticao(texto)].add(pagina)
    return {
        chave for chave, paginas in paginas_por_chave.items() if len(paginas) > limiar * n_paginas and len(paginas) >= 2
    }


def limpar_texto(t: str) -> str:
    """Limpa o texto de um bloco.

    Junta hifenização de fim de linha (``libera-\\nção`` -> ``liberação``), troca quebras simples por espaço,
    mantém quebras duplas, colapsa espaços, remove caracteres de controle e desfaz ligaduras tipográficas.
    """
    t = unicodedata.normalize("NFC", t).translate(_LIGADURAS)
    t = t.replace("\r\n", "\n").replace("\r", "\n").replace("\xa0", " ").replace("\t", " ")
    t = t.replace("\xad\n", "").replace("\xad", "")
    t = _CONTROLE.sub("", t)
    t = re.sub(r"(\w)-[ ]*\n[ ]*(?=[a-zà-öø-ÿ])", r"\1", t)
    t = re.sub(r"(?<=\w)/[ ]*\n[ ]*(?=\w)", "/", t)  # "km/" + "h" na linha seguinte
    partes = re.split(r"\n[ ]*\n", t)
    partes = [re.sub(r" {2,}", " ", re.sub(r"[ ]*\n[ ]*", " ", p)).strip() for p in partes]
    return "\n\n".join(p for p in partes if p)


def eh_titulo(t: str, regex: str) -> tuple[bool, str, str]:
    """Aplica a regex de título.

    Args:
        t: texto (uma linha) candidato a título.
        regex: expressão com a numeração no grupo 1, por exemplo ``^(\\d+(\\.\\d+){0,3})\\s+[A-Z]...``.

    Returns:
        ``(True, tema_id, tema)`` se casar, com ``tema_id`` = numeração (``"4.2"``) e ``tema`` = título
        completo; senão ``(False, "", "")``.
    """
    t = t.strip()
    m = re.match(regex, t)
    if not m:
        return False, "", ""
    return True, (m.group(1) if m.groups() else "") or "", t


def _partes(bruto: str, regex: str) -> list[tuple[tuple[str, str] | None, list[str], list[str]]]:
    """Divide um bloco nas quebras duplas e separa um eventual título no início de cada parte.

    Entradas de sumário são descartadas. O título é a primeira linha que casa com a regex; se o bloco tiver só
    duas linhas e as duas juntas também casarem, é um título quebrado em duas linhas.

    Returns:
        ``[((tema_id, tema) ou None, linhas_do_resto, todas_as_linhas)]``.
    """
    saida = []
    for parte in re.split(r"\n\s*\n", bruto):
        linhas = [ln.strip() for ln in parte.split("\n") if ln.strip()]
        linhas = [ln for ln in linhas if not _SUMARIO.search(ln)]
        if not linhas:
            continue
        ok, tema_id, tema = eh_titulo(limpar_texto(linhas[0]), regex)
        if not ok:
            saida.append((None, linhas, linhas))
            continue
        if len(linhas) == 2:
            ok2, tid2, tema2 = eh_titulo(limpar_texto("\n".join(linhas)), regex)
            if ok2:
                saida.append(((tid2, tema2), [], linhas))
                continue
        saida.append(((tema_id, tema), linhas[1:], linhas))
    return saida


def _numeracao(tema_id: str) -> tuple[int, ...]:
    return tuple(int(x) for x in tema_id.split(".") if x.isdigit())


def titulo_plausivel(novo: str, atual: str | None, max_salto: int = MAX_SALTO_NUMERACAO) -> bool:
    """Confere se a numeração ``novo`` pode suceder a seção ``atual``.

    Aceita avançar em algum nível com salto de até ``max_salto`` (``2.3`` -> ``2.4``, ``3``, ``3.1``) ou descer
    para subseções (``2.3`` -> ``2.3.1``); níveis abertos depois do ponto de avanço devem começar baixos
    (até ``max_salto``), o que admite capítulos sem título próprio (``1`` -> ``2.1``). Rejeita, por exemplo,
    passos numerados (``13.1`` -> ``1``) e linhas de tabela (``2.3`` -> ``201``). O primeiro título do
    documento é sempre aceito.
    """
    if not atual:
        return True
    n, a = _numeracao(novo), _numeracao(atual)
    if not n or not a:
        return False
    comum = 0
    while comum < min(len(n), len(a)) and n[comum] == a[comum]:
        comum += 1
    if comum == len(n):  # repete a seção atual ou um ancestral dela
        return False
    if comum < len(a) and not 0 < n[comum] - a[comum] <= max_salto:
        return False
    inicio = comum + 1 if comum < len(a) else comum
    return all(1 <= x <= max_salto for x in n[inicio:])


def _verificar_camada_texto(blocos: list[tuple[int, str]], n_paginas: int) -> None:
    chars = defaultdict(int)
    for pagina, texto in blocos:
        chars[pagina] += len(texto.strip())
    sem_texto = sum(1 for p in range(1, n_paginas + 1) if chars[p] < MIN_CHARS_PAGINA)
    if n_paginas == 0 or sem_texto > FRAC_PAGINAS_SEM_TEXTO * n_paginas:
        raise ErroUsuario("PDF sem texto; rode OCR local (ocrmypdf) antes")


def construir_paragrafos(pdf_path: Path, cfg: Config, info: dict | None = None) -> list[Paragrafo]:
    """Passos 1 e 2 completos: PDF -> parágrafos com tema e página.

    Args:
        pdf_path: caminho do PDF.
        cfg: :class:`qa_manual.config.Config` (usa ``cfg.ingestao``).
        info: se informado, recebe contagens da extração (páginas, blocos removidos, títulos etc.).

    Returns:
        Parágrafos na ordem do documento, incluindo os títulos (``eh_titulo=True``).

    Raises:
        ErroUsuario: se o PDF não abrir ou não tiver camada de texto.
    """
    ing = cfg.ingestao
    blocos, n_paginas = _extrair(Path(pdf_path))
    _verificar_camada_texto(blocos, n_paginas)

    n_blocos = len(blocos)
    if ing.remover_cabecalhos_rodapes:
        repetidos = detectar_cabecalhos_rodapes(blocos, n_paginas, ing.limiar_repeticao_cabecalho)
        blocos = [(p, t) for p, t in blocos if _chave_repeticao(t) not in repetidos]
    removidos = n_blocos - len(blocos)

    paragrafos: list[Paragrafo] = []
    tema, tema_id = TEMA_INICIAL, ""
    curtos = rejeitados = 0

    def adicionar(pagina: int, texto: str, titulo: bool) -> None:
        paragrafos.append(
            Paragrafo(idx=len(paragrafos), pagina=pagina, tema=tema, tema_id=tema_id, texto=texto, eh_titulo=titulo)
        )

    for pagina, bruto in blocos:
        for titulo, resto, linhas in _partes(bruto, ing.regex_titulo):
            if titulo and ing.validar_sequencia_titulos and not titulo_plausivel(titulo[0], tema_id):
                rejeitados += 1
                titulo, resto = None, linhas
            if titulo:
                tema_id, tema = titulo
                adicionar(pagina, tema, True)
            for texto in limpar_texto("\n".join(resto)).split("\n\n"):
                if len(texto) < ing.min_chars_paragrafo:
                    curtos += bool(texto)
                    continue
                adicionar(pagina, texto, False)

    n_titulos = sum(p.eh_titulo for p in paragrafos)
    log.info(
        "ingestão: %d páginas, %d blocos, %d cabeçalhos/rodapés removidos, %d títulos (%d candidatos fora de "
        "sequência tratados como texto), %d parágrafos, %d curtos descartados",
        n_paginas,
        n_blocos,
        removidos,
        n_titulos,
        rejeitados,
        len(paragrafos) - n_titulos,
        curtos,
    )
    if info is not None:
        info.update(
            n_paginas=n_paginas,
            n_blocos=n_blocos,
            blocos_cabecalho_rodape_removidos=removidos,
            n_titulos=n_titulos,
            titulos_fora_de_sequencia=rejeitados,
            n_paragrafos=len(paragrafos) - n_titulos,
            paragrafos_curtos_descartados=curtos,
        )
    if not any(not p.eh_titulo for p in paragrafos):
        raise ErroUsuario(f"Nenhum parágrafo extraído de {pdf_path}; confira o PDF e ingestao.min_chars_paragrafo.")
    return paragrafos


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/chunking.py
"""Passo 3: agrupamento de parágrafos em trechos de até ``max_tokens``, sem cruzar temas e sem cortar parágrafos."""

from __future__ import annotations

import statistics
from collections import Counter
from dataclasses import asdict, dataclass
from typing import TYPE_CHECKING

from .ingest import Paragrafo

if TYPE_CHECKING:
    from .config import Config

N_FAIXAS_HISTOGRAMA = 10


@dataclass
class Trecho:
    """Unidade de recuperação gravada em ``data/trechos.jsonl`` (contrato 5.1)."""

    id: str
    tema: str
    tema_id: str
    pagina_inicio: int
    pagina_fim: int
    paragrafos: int
    n_tokens: int
    texto: str

    @classmethod
    def de_dict(cls, d: dict) -> Trecho:
        """Constrói a partir de um registro JSONL."""
        return cls(**{k: d[k] for k in cls.__dataclass_fields__})

    def como_dict(self) -> dict:
        """Registro JSONL."""
        return asdict(self)


def estimar_tokens(texto: str, tokens_por_palavra: float) -> int:
    """Estimativa de tokens a partir do número de palavras (1,3 token por palavra em português)."""
    return max(1, round(len(texto.split()) * tokens_por_palavra))


def _tokens(n_palavras: int, tokens_por_palavra: float) -> int:
    return max(1, round(n_palavras * tokens_por_palavra))


def _segmentos_por_tema(paragrafos: list[Paragrafo]) -> list[list[Paragrafo]]:
    """Sequências contíguas de parágrafos (sem títulos) sob o mesmo título."""
    segmentos: list[list[Paragrafo]] = []
    novo = True
    for p in paragrafos:
        if p.eh_titulo:
            novo = True
            continue
        if novo or p.tema_id != segmentos[-1][-1].tema_id or p.tema != segmentos[-1][-1].tema:
            segmentos.append([])
            novo = False
        segmentos[-1].append(p)
    return segmentos


def _agrupar_segmento(seg: list[Paragrafo], max_tokens: int, min_tokens: int, tpp: float) -> list[list[Paragrafo]]:
    # Regras 3 e 4: acumula enquanto couber; um parágrafo maior que max_tokens fica sozinho.
    grupos: list[list[Paragrafo]] = []
    atual: list[Paragrafo] = []
    palavras = 0
    for p in seg:
        n = len(p.texto.split())
        if atual and _tokens(palavras + n, tpp) > max_tokens:
            grupos.append(atual)
            atual, palavras = [], 0
        atual.append(p)
        palavras += n
    if atual:
        grupos.append(atual)

    # Regra 5: grupo abaixo de min_tokens é fundido com o seguinte; o último do tema fica como está.
    fundidos: list[list[Paragrafo]] = []
    pendente: list[Paragrafo] = []
    for i, g in enumerate(grupos):
        g = pendente + g
        pendente = []
        ultimo = i == len(grupos) - 1
        if not ultimo and _tokens(sum(len(p.texto.split()) for p in g), tpp) < min_tokens:
            pendente = g
            continue
        fundidos.append(g)
    return fundidos


def agrupar_trechos(paragrafos: list[Paragrafo], cfg: Config) -> list[Trecho]:
    """Agrupa parágrafos em trechos.

    Regras:
        1. Percorre os parágrafos em ordem; títulos não entram no texto.
        2. Um trecho só contém parágrafos do mesmo tema.
        3. Acumula parágrafos enquanto ``tokens_acumulados + tokens_do_proximo <= max_tokens``.
        4. Um parágrafo maior que ``max_tokens`` vira um trecho sozinho (nunca é cortado).
        5. Um trecho com menos de ``min_tokens`` é fundido com o próximo do mesmo tema; se for o último do
           tema, fica como está.
        6. Se ``sobreposicao_paragrafos`` > 0, cada novo trecho começa repetindo os últimos N parágrafos do
           anterior (mesmo tema).
        7. Ids ``t0001``, ``t0002``, ... na ordem do documento.

    Args:
        paragrafos: saída de :func:`qa_manual.ingest.construir_paragrafos`.
        cfg: :class:`qa_manual.config.Config` (usa ``cfg.trechos``).

    Returns:
        Trechos na ordem do documento; os parágrafos de um trecho são unidos por quebra de linha.
    """
    tc = cfg.trechos
    trechos: list[Trecho] = []
    for seg in _segmentos_por_tema(paragrafos):
        grupos = _agrupar_segmento(seg, tc.max_tokens, tc.min_tokens, tc.tokens_por_palavra)
        n = tc.sobreposicao_paragrafos
        if n > 0:
            grupos = [grupos[0]] + [grupos[i - 1][-n:] + grupos[i] for i in range(1, len(grupos))]
        for g in grupos:
            texto = "\n".join(p.texto for p in g)
            trechos.append(
                Trecho(
                    id=f"t{len(trechos) + 1:04d}",
                    tema=g[0].tema,
                    tema_id=g[0].tema_id,
                    pagina_inicio=min(p.pagina for p in g),
                    pagina_fim=max(p.pagina for p in g),
                    paragrafos=len(g),
                    n_tokens=estimar_tokens(texto, tc.tokens_por_palavra),
                    texto=texto,
                )
            )
    return trechos


def estatisticas(trechos: list[Trecho], cfg: Config, info: dict | None = None) -> dict:
    """Estatísticas de ``data/trechos_stats.json``.

    Contém número de trechos, média/mediana/máximo de tokens, número de temas, trechos por tema (chave =
    numeração do tema, sem o título, para não expor conteúdo) e histograma de tokens em 10 faixas.
    """
    tokens = [t.n_tokens for t in trechos] or [0]
    maximo = max(tokens)
    largura = max(1, -(-maximo // N_FAIXAS_HISTOGRAMA))  # teto
    histograma = Counter(min(t // largura, N_FAIXAS_HISTOGRAMA - 1) for t in tokens)
    faixas = {f"{i * largura}-{(i + 1) * largura - 1}": histograma.get(i, 0) for i in range(N_FAIXAS_HISTOGRAMA)}
    por_tema = Counter(t.tema_id or "sem_numero" for t in trechos)
    return {
        **(info or {}),
        "n_trechos": len(trechos),
        "n_temas": len(por_tema),
        "tokens": {
            "media": round(statistics.mean(tokens), 1),
            "mediana": statistics.median(tokens),
            "minimo": min(tokens),
            "maximo": maximo,
        },
        "acima_de_max_tokens": sum(t > cfg.trechos.max_tokens for t in tokens),
        "abaixo_de_min_tokens": sum(t < cfg.trechos.min_tokens for t in tokens),
        "trechos_por_tema": dict(por_tema),
        "histograma_tokens": faixas,
        "params_trechos": asdict(cfg.trechos),
    }


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/tokenize_pt.py
"""Tokenização em português para o BM25 (a mesma função é usada na indexação e na consulta)."""

from __future__ import annotations

import re
import unicodedata
from functools import lru_cache
from typing import TYPE_CHECKING

from .io_utils import ErroUsuario

if TYPE_CHECKING:
    from .config import Config

_NAO_ALFANUMERICO = re.compile(r"[\W_]+")


def remover_acentos(texto: str) -> str:
    """Remove diacríticos (decomposição NFKD sem marcas combinantes)."""
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))


@lru_cache(maxsize=2)
def stopwords_pt(manter_acentos: bool = True) -> frozenset[str]:
    """Stop-words do NLTK para português, carregadas uma vez por processo.

    Raises:
        ErroUsuario: se o corpus ``stopwords`` do NLTK não estiver instalado.
    """
    try:
        from nltk.corpus import stopwords

        palavras = stopwords.words("portuguese")
    except LookupError:
        raise ErroUsuario(
            "Lista de stop-words do NLTK ausente; rode: python -c \"import nltk; nltk.download('stopwords')\""
        ) from None
    palavras = [unicodedata.normalize("NFC", p.lower()) for p in palavras]
    if not manter_acentos:
        palavras = [remover_acentos(p) for p in palavras]
    return frozenset(palavras)


def tokenizar(texto: str, remover_stopwords: bool = True, manter_acentos: bool = True) -> list[str]:
    """Tokeniza para o índice léxico.

    Passos: minúsculas; remoção de diacríticos se ``manter_acentos=False``; tudo que não for letra ou dígito
    vira espaço; divisão em espaços; remoção de tokens de 1 caractere; remoção das stop-words do NLTK.

    Args:
        texto: texto de entrada.
        remover_stopwords: remove stop-words do português.
        manter_acentos: preserva acentos (``liberação`` e ``liberacao`` são tokens distintos).

    Returns:
        Lista de tokens.
    """
    t = unicodedata.normalize("NFC", texto.lower())
    if not manter_acentos:
        t = remover_acentos(t)
    tokens = [tok for tok in _NAO_ALFANUMERICO.sub(" ", t).split() if len(tok) > 1]
    if remover_stopwords:
        sw = stopwords_pt(manter_acentos)
        tokens = [tok for tok in tokens if tok not in sw]
    return tokens


def tokenizar_cfg(texto: str, cfg: Config) -> list[str]:
    """:func:`tokenizar` com as opções de ``cfg.bm25``."""
    return tokenizar(texto, cfg.bm25.remover_stopwords, cfg.bm25.manter_acentos)


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/prompts.py
"""Templates de prompt em ``prompts/``: carga e preenchimento com ``str.format``.

Chaves literais nos arquivos (JSON de exemplo, formato da citação) são escritas como ``{{`` e ``}}``.
"""

from __future__ import annotations

import string
from functools import cache
from pathlib import Path

from .io_utils import RAIZ, ErroUsuario

DIR_PROMPTS = RAIZ / "prompts"

# Placeholders esperados em cada template (conferidos nos testes).
PLACEHOLDERS = {
    "leitor": {"trechos", "pergunta"},
    "leitor_s0": {"pergunta"},
    "gerador_perguntas": {"tipo", "texto"},
    "gerador_sem_resposta": {"texto"},
    "gerador_pares_treino": {"tipo", "tema_id", "pagina", "texto"},
}


@cache
def carregar(nome: str) -> str:
    """Lê ``prompts/<nome>.txt`` (o sufixo ``.txt`` é opcional).

    Raises:
        ErroUsuario: se o arquivo não existir.
    """
    arquivo = DIR_PROMPTS / (nome if nome.endswith(".txt") else f"{nome}.txt")
    if not arquivo.is_file():
        raise ErroUsuario(f"Prompt não encontrado: {arquivo}")
    return arquivo.read_text(encoding="utf-8")


def placeholders(template: str) -> set[str]:
    """Nomes dos campos ``{campo}`` do template (chaves escapadas não contam)."""
    return {campo for _, campo, _, _ in string.Formatter().parse(template) if campo}


def preencher(template: str, **campos: object) -> str:
    """Preenche o template com ``str.format``; valores com chaves não são reprocessados.

    Raises:
        ErroUsuario: se faltar algum campo do template.
    """
    try:
        return template.format(**campos)
    except KeyError as e:
        raise ErroUsuario(f"Campo {e} ausente ao preencher o prompt.") from None


def caminho(nome: str) -> Path:
    """Caminho do arquivo de um prompt."""
    return DIR_PROMPTS / f"{nome}.txt"


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/generate.py
"""Passo 10: montagem do prompt do leitor e geração da resposta pelo Ollama."""

from __future__ import annotations

import logging
import re
import time
from typing import TYPE_CHECKING

from . import prompts
from .chunking import Trecho
from .ollama_client import OllamaClient

if TYPE_CHECKING:
    from .config import Config

log = logging.getLogger(__name__)

_THINK = re.compile(r"<think>.*?(?:</think>|$)\s*", re.DOTALL)


def rotulo_secao(t: Trecho) -> str:
    """Seção exibida ao leitor: a numeração do tema ou, sem numeração, o próprio título."""
    return t.tema_id or t.tema


def formatar_trechos(trechos: list[Trecho]) -> str:
    """Renderiza cada trecho como ``[n] (seção {tema_id}, p. {pagina_inicio}) {texto}``."""
    return "\n\n".join(
        f"[{n}] (seção {rotulo_secao(t)}, p. {t.pagina_inicio}) {t.texto}" for n, t in enumerate(trechos, start=1)
    )


def montar_prompt(pergunta: str, trechos: list[Trecho]) -> str:
    """Prompt do leitor: ``leitor.txt`` com trechos ou ``leitor_s0.txt`` quando não há trechos (S0)."""
    if not trechos:
        return prompts.preencher(prompts.carregar("leitor_s0"), pergunta=pergunta.strip())
    return prompts.preencher(prompts.carregar("leitor"), trechos=formatar_trechos(trechos), pergunta=pergunta.strip())


def remover_think(texto: str) -> tuple[str, bool]:
    """Remove blocos ``<think>...</think>`` (também um bloco aberto e não fechado). Devolve ``(texto, removeu)``."""
    if "<think>" not in texto and "</think>" not in texto:
        return texto, False
    texto = _THINK.sub("", texto)
    if "</think>" in texto:  # fechamento sem abertura: o raciocínio é tudo o que vem antes
        texto = texto.split("</think>", 1)[1]
    return texto, True


def responder(pergunta: str, trechos: list[Trecho], leitor: str, cfg: Config, client: OllamaClient) -> tuple[str, dict]:
    """Gera a resposta do leitor.

    Args:
        pergunta: pergunta do usuário.
        trechos: trechos recuperados (lista vazia em S0).
        leitor: ``base`` ou ``ajustado``.
        cfg: configuração (modelos e decodificação).
        client: cliente Ollama.

    Returns:
        ``(resposta, {'t_leitura_s', 'n_tokens_prompt', 'n_tokens_resposta'})``.
    """
    modelo = cfg.modelo_leitor(leitor)
    prompt = montar_prompt(pergunta, trechos)
    t0 = time.perf_counter()
    r = client.chat(
        model=modelo,
        messages=[{"role": "user", "content": prompt}],
        options=cfg.decodificacao.opcoes(),
        think=cfg.decodificacao.think,
    )
    t_leitura = time.perf_counter() - t0
    texto, removeu = remover_think(r.get("content", ""))
    if removeu:
        log.warning("bloco <think> removido da resposta de %s apesar de think=%s", modelo, cfg.decodificacao.think)
    return texto.strip(), {
        "t_leitura_s": round(t_leitura, 4),
        "n_tokens_prompt": r.get("prompt_eval_count"),
        "n_tokens_resposta": r.get("eval_count"),
    }


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/verify.py
"""Passo 11: detecção de abstenção, extração e validação das citações ``(seção X, p. N)``."""

from __future__ import annotations

import re

from .chunking import Trecho
from .tokenize_pt import remover_acentos

_SECAO = r"se[çc][ãa]o\s*\{?\s*(?P<secao>\d+(?:\.\d+)*)\s*\}?"
_PAGINA = r"(?:p\.|p[áa]g\.?|p[áa]gina)\s*\{?\s*(?P<pagina>\d+)\s*\}?"
_PAR = re.compile(_SECAO + r"\s*[,;–-]?\s*" + _PAGINA, re.IGNORECASE)
_SO_SECAO = re.compile(_SECAO, re.IGNORECASE)
_SO_PAGINA = re.compile(r"(?<![\w.])" + _PAGINA, re.IGNORECASE)
# Parênteses que contêm uma citação: "(seção 4.2, p. 17)", "(fonte: seção 2)", "(p. 3)".
_PARENTESES_CITACAO = re.compile(r"\(\s*[^()]*?(?:se[çc][ãa]o|\bp\.|\bp[áa]g)[^()]*\)", re.IGNORECASE)


def _normalizar_abstencao(t: str) -> str:
    t = remover_acentos(t.lower())
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()


def detectar_abstencao(resposta: str, frase: str) -> bool:
    """True se a resposta contém a frase de abstenção, ignorando maiúsculas, acentos e pontuação."""
    alvo = _normalizar_abstencao(frase)
    return bool(alvo) and alvo in _normalizar_abstencao(resposta)


def extrair_citacoes(resposta: str) -> list[dict]:
    """Extrai citações nos formatos ``(seção 4.2, p. 17)``, ``seção 4.2``, ``p. 17``, ``pág. 17`` e ``página 17``.

    Returns:
        ``[{'tema_id': '4.2', 'pagina': 17}, ...]`` na ordem em que aparecem; o campo ausente fica ``None``.
    """
    achados: list[tuple[int, dict]] = []
    ocupado: list[tuple[int, int]] = []
    for m in _PAR.finditer(resposta):
        achados.append((m.start(), {"tema_id": m.group("secao"), "pagina": int(m.group("pagina"))}))
        ocupado.append(m.span())

    def livre(m: re.Match) -> bool:
        return not any(a <= m.start() < b for a, b in ocupado)

    for m in _SO_SECAO.finditer(resposta):
        if livre(m):
            achados.append((m.start(), {"tema_id": m.group("secao"), "pagina": None}))
    for m in _SO_PAGINA.finditer(resposta):
        if livre(m):
            achados.append((m.start(), {"tema_id": None, "pagina": int(m.group("pagina"))}))
    return [c for _, c in sorted(achados, key=lambda x: x[0])]


def _casa(citacao: dict, t: Trecho) -> bool:
    tid, pag = citacao.get("tema_id"), citacao.get("pagina")
    if tid is None and pag is None:
        return False
    if tid is not None and tid != t.tema_id:
        return False
    return pag is None or t.pagina_inicio <= pag <= t.pagina_fim


def validar_fonte(citacoes: list[dict], recuperadas: list[Trecho]) -> bool:
    """True se alguma citação coincide com um trecho recuperado.

    Coincidir significa: mesma seção (``tema_id``) e página dentro de ``[pagina_inicio, pagina_fim]``. Citações
    parciais (só seção ou só página) são comparadas pelo campo presente. Sem trechos (S0), devolve False.
    """
    return any(_casa(c, t) for c in citacoes for t in recuperadas)


def remover_citacoes(resposta: str) -> str:
    """Resposta sem as citações de fonte, para comparar com a referência (EM, F1, BERTScore)."""
    texto = _PARENTESES_CITACAO.sub(" ", resposta)
    texto = _PAR.sub(" ", texto)
    texto = re.sub(r"\s+([.,;:!?])", r"\1", texto)
    return re.sub(r"\s{2,}", " ", texto).strip(" \n-–")


In [ ]:
%%writefile /content/qa_manual_colab/qa_manual/gold.py
"""Passo 12: conjunto de avaliação (gold).

Fluxo: candidatas geradas pelo LLM local (``gerar_candidatas``) -> CSV para revisão humana
(``exportar_revisao``) -> importação das decisões (``importar_revisao``) -> divisão dev/teste por tema
(``dividir``). ``reservar_ppl`` separa trechos para a medida de perplexidade.
"""

from __future__ import annotations

import csv
import json
import logging
import math
import random
import re
from collections import defaultdict
from pathlib import Path
from typing import TYPE_CHECKING

from tqdm import tqdm

from . import prompts
from .chunking import Trecho
from .io_utils import ErroUsuario, sha256_texto
from .ollama_client import OllamaClient

if TYPE_CHECKING:
    from .config import Config

log = logging.getLogger(__name__)

TIPOS = ("factual", "procedimental", "sem_resposta")
DECISOES = ("aceitar", "editar", "descartar")
COLUNAS_REVISAO = [
    "id",
    "tipo",
    "tema_id",
    "evidencia",
    "pergunta",
    "resposta_ref",
    "texto_evidencia",
    "decisao",
    "pergunta_corrigida",
    "resposta_corrigida",
    "observacao",
]
CAMPOS_GOLD = ("id", "pergunta", "resposta_ref", "evidencia", "tipo")
MIN_TOKENS_AMOSTRA = 120
MAX_TENTATIVAS = 3
SEPARADOR_EVIDENCIA = "|"
_OBJETO_JSON = re.compile(r"\{.*\}", re.DOTALL)
_ID_TRECHO = re.compile(r"t\d+")


# ---------------------------------------------------------------------------
# Amostragem e geração
# ---------------------------------------------------------------------------


def amostrar_trechos(trechos: list[Trecho], n: int, seed: int) -> list[Trecho]:
    """Amostra estratificada por ``tema_id``.

    A cota de cada tema é proporcional ao número de trechos dele, com pelo menos um trecho por tema quando
    ``n`` comporta todos os temas (método do maior resto). Trechos com menos de 120 tokens são excluídos.

    Returns:
        Até ``n`` trechos distintos, em ordem aleatória determinística.
    """
    rng = random.Random(seed)
    grupos: dict[str, list[Trecho]] = defaultdict(list)
    for t in trechos:
        if t.n_tokens >= MIN_TOKENS_AMOSTRA:
            grupos[t.tema_id].append(t)
    total = sum(len(g) for g in grupos.values())
    n = min(n, total)
    if n <= 0:
        return []
    temas = sorted(grupos)
    base = {t: 1 if n >= len(temas) else 0 for t in temas}
    resto = n - sum(base.values())
    capacidade = sum(len(grupos[t]) - base[t] for t in temas)
    ideal = {t: resto * (len(grupos[t]) - base[t]) / capacidade if capacidade else 0.0 for t in temas}
    cota = {t: base[t] + math.floor(ideal[t]) for t in temas}
    sorteio = {t: rng.random() for t in temas}
    ordem = sorted(temas, key=lambda t: (-(ideal[t] - math.floor(ideal[t])), sorteio[t]))
    faltam = n - sum(cota.values())
    while faltam > 0:
        for t in ordem:
            if faltam and cota[t] < len(grupos[t]):
                cota[t] += 1
                faltam -= 1
    escolhidos = [tr for t in temas for tr in rng.sample(grupos[t], cota[t])]
    rng.shuffle(escolhidos)
    return escolhidos


def extrair_json(texto: str) -> dict | None:
    """Extrai o primeiro objeto JSON da saída do gerador, tolerando texto antes e depois."""
    m = _OBJETO_JSON.search(texto)
    if not m:
        return None
    try:
        dados = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return dados if isinstance(dados, dict) else None


def gerar_json(
    client: OllamaClient, modelo: str, prompt: str, chaves: tuple[str, ...], cfg: Config, contagem: dict
) -> dict | None:
    """Chama o gerador até ``MAX_TENTATIVAS`` vezes até obter um JSON com ``chaves`` preenchidas.

    A cada tentativa a seed muda (com temperatura 0 e a mesma seed, a saída seria idêntica). ``contagem``
    acumula ``tentativas`` e ``validas``.
    """
    for tentativa in range(MAX_TENTATIVAS):
        opcoes = cfg.decodificacao.opcoes() | {"seed": cfg.gold.seed + tentativa, "num_predict": 256}
        r = client.chat(model=modelo, messages=[{"role": "user", "content": prompt}], options=opcoes, format="json")
        contagem["tentativas"] = contagem.get("tentativas", 0) + 1
        dados = extrair_json(r.get("content", ""))
        if dados and all(isinstance(dados.get(c), str) and dados[c].strip() for c in chaves):
            contagem["validas"] = contagem.get("validas", 0) + 1
            return {c: dados[c].strip() for c in chaves}
    return None


def gerar_candidatas(
    cfg: Config, client: OllamaClient, trechos: list[Trecho], n: int | None = None
) -> tuple[list[dict], dict]:
    """Gera perguntas candidatas com o modelo ``modelos.gerador``.

    ``n * (1 - frac_sem_resposta)`` trechos recebem uma pergunta respondível (60% factual, 40% procedimental)
    e ``n * frac_sem_resposta`` recebem uma pergunta sem resposta no trecho. Saídas inválidas são descartadas
    depois de três tentativas.

    Returns:
        ``(candidatas, estatisticas)`` com candidatas no contrato 5.3 (mais ``trecho_origem``).
    """
    n = n or cfg.gold.n_candidatas
    n_sem = round(n * cfg.gold.frac_sem_resposta)
    n_resp = n - n_sem
    respondiveis = amostrar_trechos(trechos, n_resp, cfg.gold.seed)
    sem_resposta = amostrar_trechos(trechos, n_sem, cfg.gold.seed + 1)
    if len(respondiveis) < n_resp:
        log.warning(
            "só %d trechos com >= %d tokens; menos candidatas que o pedido", len(respondiveis), MIN_TOKENS_AMOSTRA
        )

    tarefas = [("procedimental" if i % 5 in (1, 3) else "factual", t) for i, t in enumerate(respondiveis)]
    tarefas += [("sem_resposta", t) for t in sem_resposta]
    modelo = cfg.modelos.gerador
    contagem: dict = {}
    candidatas: list[dict] = []
    for tipo, t in tqdm(tarefas, desc="candidatas", unit="pergunta"):
        if tipo == "sem_resposta":
            prompt = prompts.preencher(prompts.carregar("gerador_sem_resposta"), texto=t.texto)
            dados = gerar_json(client, modelo, prompt, ("pergunta",), cfg, contagem)
        else:
            prompt = prompts.preencher(prompts.carregar("gerador_perguntas"), tipo=tipo, texto=t.texto)
            dados = gerar_json(client, modelo, prompt, ("pergunta", "resposta"), cfg, contagem)
        if dados is None:
            log.warning("gerador sem JSON válido para o trecho %s após %d tentativas", t.id, MAX_TENTATIVAS)
            continue
        candidatas.append(
            {
                "id": f"q{len(candidatas) + 1:03d}",
                "pergunta": dados["pergunta"],
                "resposta_ref": cfg.abstencao.frase if tipo == "sem_resposta" else dados["resposta"],
                "evidencia": [] if tipo == "sem_resposta" else [t.id],
                "tipo": tipo,
                "tema_id": t.tema_id,
                "origem": "gerado",
                "revisado": False,
                "trecho_origem": t.id,
            }
        )
    tentativas = contagem.get("tentativas", 0)
    stats = {
        "pedidas": n,
        "geradas": len(candidatas),
        "tentativas": tentativas,
        "json_valido_pct": round(contagem.get("validas", 0) / tentativas, 3) if tentativas else 0.0,
    }
    log.info("candidatas: %s", stats)
    return candidatas, stats


# ---------------------------------------------------------------------------
# Revisão humana (CSV)
# ---------------------------------------------------------------------------


def exportar_revisao(candidatas: list[dict], trechos: list[Trecho], destino: Path) -> int:
    """Grava o CSV de revisão (contrato 5.4) com o texto do trecho-evidência para o revisor.

    O arquivo usa ``;`` como separador e UTF-8 com BOM, para abrir direto no Excel em português e no
    LibreOffice. Em itens ``sem_resposta``, ``texto_evidencia`` mostra o trecho que originou a pergunta.

    Returns:
        Número de linhas gravadas.
    """
    por_id = {t.id: t for t in trechos}
    Path(destino).parent.mkdir(parents=True, exist_ok=True)
    with open(destino, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS_REVISAO, delimiter=";", quoting=csv.QUOTE_MINIMAL)
        w.writeheader()
        for c in candidatas:
            ids = c.get("evidencia") or ([c["trecho_origem"]] if c.get("trecho_origem") else [])
            textos = [por_id[i].texto for i in ids if i in por_id]
            prefixo = "[trecho de origem; a resposta NÃO deve estar no manual] " if c["tipo"] == "sem_resposta" else ""
            w.writerow(
                {
                    "id": c["id"],
                    "tipo": c["tipo"],
                    "tema_id": c.get("tema_id", ""),
                    "evidencia": SEPARADOR_EVIDENCIA.join(c.get("evidencia", [])),
                    "pergunta": c["pergunta"],
                    "resposta_ref": c["resposta_ref"],
                    "texto_evidencia": prefixo + "\n---\n".join(textos),
                    "decisao": "",
                    "pergunta_corrigida": "",
                    "resposta_corrigida": "",
                    "observacao": "",
                }
            )
    return len(candidatas)


def _ler_csv(caminho: Path) -> list[dict]:
    with open(caminho, encoding="utf-8-sig", newline="") as f:
        amostra = f.read(4096)
        f.seek(0)
        primeira = amostra.split("\n", 1)[0]
        delimitador = max(";,\t", key=primeira.count)
        leitor = csv.DictReader(f, delimiter=delimitador)
        faltando = [c for c in COLUNAS_REVISAO if c not in (leitor.fieldnames or [])]
        if faltando:
            raise ErroUsuario(f"{caminho}: coluna(s) ausente(s): {', '.join(faltando)}.")
        return list(leitor)


def importar_revisao(csv_path: Path, frase_abstencao: str) -> list[dict]:
    """Aplica as decisões do CSV revisado.

    ``aceitar`` mantém o item; ``editar`` substitui pergunta e/ou resposta pelas colunas corrigidas;
    ``descartar`` remove. Todos os itens mantidos ficam com ``revisado: true``.

    Raises:
        ErroUsuario: se alguma linha estiver sem decisão, com decisão inválida ou inconsistente.
    """
    linhas = _ler_csv(Path(csv_path))
    sem_decisao, erros, itens, vistos = [], [], [], set()
    for n, row in enumerate(linhas, start=2):
        qid = (row.get("id") or "").strip()
        decisao = (row.get("decisao") or "").strip().lower()
        if not decisao:
            sem_decisao.append(qid or f"linha {n}")
            continue
        if decisao not in DECISOES:
            erros.append(f"linha {n} ({qid}): decisão '{decisao}' inválida; use {', '.join(DECISOES)}")
            continue
        if qid in vistos:
            erros.append(f"linha {n}: id {qid} repetido")
            continue
        vistos.add(qid)
        if decisao == "descartar":
            continue
        tipo = (row.get("tipo") or "").strip()
        if tipo not in TIPOS:
            erros.append(f"linha {n} ({qid}): tipo '{tipo}' inválido; use {', '.join(TIPOS)}")
            continue
        pergunta, resposta = row["pergunta"].strip(), row["resposta_ref"].strip()
        if decisao == "editar":
            pc, rc = row.get("pergunta_corrigida", "").strip(), row.get("resposta_corrigida", "").strip()
            if not pc and not rc:
                erros.append(f"linha {n} ({qid}): 'editar' sem pergunta_corrigida nem resposta_corrigida")
                continue
            pergunta, resposta = pc or pergunta, rc or resposta
        evidencia = _ID_TRECHO.findall(row.get("evidencia") or "")
        if tipo == "sem_resposta":
            resposta, evidencia = frase_abstencao, []
        elif not evidencia:
            erros.append(f"linha {n} ({qid}): item {tipo} sem evidência")
            continue
        itens.append(
            {
                "id": qid,
                "pergunta": pergunta,
                "resposta_ref": resposta,
                "evidencia": evidencia,
                "tipo": tipo,
                "tema_id": (row.get("tema_id") or "").strip(),
                "origem": "editado" if decisao == "editar" else "gerado",
                "revisado": True,
            }
        )
    if sem_decisao:
        exemplos = ", ".join(sem_decisao[:10]) + (" ..." if len(sem_decisao) > 10 else "")
        erros.insert(0, f"{len(sem_decisao)} linha(s) sem decisão: {exemplos}")
    if erros:
        raise ErroUsuario(f"{csv_path}: revisão incompleta ou inválida:\n  - " + "\n  - ".join(erros))
    log.info("revisão importada: %d itens mantidos de %d linhas", len(itens), len(linhas))
    return itens


# ---------------------------------------------------------------------------
# Divisão dev/teste e reserva para perplexidade
# ---------------------------------------------------------------------------


def _temas_dev(por_tema: dict[str, list[dict]], alvo: float, rng: random.Random) -> set[str]:
    temas = sorted(por_tema)
    rng.shuffle(temas)
    escolhidos, n_dev = set(), 0
    for t in temas:
        tam = len(por_tema[t])
        if abs(n_dev + tam - alvo) < abs(n_dev - alvo):
            escolhidos.add(t)
            n_dev += tam
    return escolhidos


def dividir(gold: list[dict], frac_dev: float, seed: int) -> tuple[list[dict], list[dict]]:
    """Divide o gold em dev e teste por ``tema_id``: todos os itens de um tema caem no mesmo split.

    Os temas são embaralhados e atribuídos ao dev até ~``frac_dev`` dos itens. Se houver itens
    ``sem_resposta``, os dois splits recebem pelo menos um; quando isso é impossível respeitando os temas,
    só os ``sem_resposta`` (que não têm evidência) são redistribuídos, com aviso no log.

    Returns:
        ``(dev, test)`` com o campo ``split`` preenchido.
    """
    if not gold:
        raise ErroUsuario("gold vazio; nada a dividir.")
    por_tema: dict[str, list[dict]] = defaultdict(list)
    for it in gold:
        por_tema[it.get("tema_id", "")].append(it)
    if len(por_tema) < 2:
        raise ErroUsuario("não é possível dividir por tema: todos os itens são do mesmo tema.")
    alvo = frac_dev * len(gold)
    tem_sem = any(it["tipo"] == "sem_resposta" for it in gold)

    def ok(dev: list[dict], test: list[dict]) -> bool:
        if not dev or not test:
            return False
        return not tem_sem or all(any(it["tipo"] == "sem_resposta" for it in s) for s in (dev, test))

    for tentativa in range(200):
        temas_dev = _temas_dev(por_tema, alvo, random.Random(seed + tentativa))
        dev = [it for it in gold if it.get("tema_id", "") in temas_dev]
        test = [it for it in gold if it.get("tema_id", "") not in temas_dev]
        if ok(dev, test):
            break
    else:
        respondiveis = [it for it in gold if it["tipo"] != "sem_resposta"]
        sem = [it for it in gold if it["tipo"] == "sem_resposta"]
        por_tema_r: dict[str, list[dict]] = defaultdict(list)
        for it in respondiveis:
            por_tema_r[it.get("tema_id", "")].append(it)
        temas_dev = _temas_dev(por_tema_r, frac_dev * len(respondiveis), random.Random(seed))
        dev = [it for it in respondiveis if it.get("tema_id", "") in temas_dev]
        test = [it for it in respondiveis if it.get("tema_id", "") not in temas_dev]
        rng = random.Random(seed)
        sem = sorted(sem, key=lambda it: it["id"])
        rng.shuffle(sem)
        n_sem_dev = min(len(sem) - 1, max(1, round(frac_dev * len(sem))))
        dev += sem[:n_sem_dev]
        test += sem[n_sem_dev:]
        log.warning("itens sem_resposta redistribuídos sem respeitar o tema (não têm evidência)")
        if not dev or not test:
            raise ErroUsuario("não foi possível dividir o gold em dev e teste não vazios.")

    dev = sorted(({**it, "split": "dev"} for it in dev), key=lambda it: it["id"])
    test = sorted(({**it, "split": "test"} for it in test), key=lambda it: it["id"])
    log.info("divisão: %d dev, %d test (%d temas)", len(dev), len(test), len(por_tema))
    return dev, test


def reservar_ppl(
    trechos: list[Trecho], gold_test: list[dict] | None, n: int = 30, seed: int = 42, destino: Path | None = None
) -> list[Trecho]:
    """Escolhe ``n`` trechos de temas que não aparecem em ``gold_test`` para a medida de perplexidade.

    Args:
        trechos: todos os trechos do manual.
        gold_test: itens do teste (``None`` se ainda não existir; então nenhum tema é excluído e há aviso).
        n: número de trechos.
        seed: semente.
        destino: se informado, grava os textos separados por linha em branco.

    Returns:
        Trechos escolhidos, em ordem de id.
    """
    if gold_test is None:
        log.warning(
            "gold de teste ainda não existe; a reserva para perplexidade não exclui temas do teste. "
            "Depois da divisão, rode `qa-manual importar-revisao --dividir --reservar-ppl %d`",
            n,
        )
    temas_teste = {it.get("tema_id", "") for it in gold_test or []}
    candidatos = [t for t in trechos if t.tema_id not in temas_teste]
    if len(candidatos) < n:
        log.warning("só %d trechos fora dos temas do teste (pedidos %d)", len(candidatos), n)
    escolhidos = sorted(random.Random(seed).sample(candidatos, min(n, len(candidatos))), key=lambda t: t.id)
    if destino is not None:
        texto = "\n\n".join(t.texto for t in escolhidos) + "\n"
        Path(destino).parent.mkdir(parents=True, exist_ok=True)
        Path(destino).write_text(texto, encoding="utf-8")
        log.info("perplexidade: %d trechos reservados (sha256 %s)", len(escolhidos), sha256_texto(texto)[:12])
    return escolhidos


def validar_gold(itens: list[dict], origem: str | Path) -> None:
    """Confere campos obrigatórios e tipos de um arquivo gold.

    Raises:
        ErroUsuario: se o arquivo estiver vazio ou algum item for inválido (a mensagem cita só ids).
    """
    if not itens:
        raise ErroUsuario(f"{origem}: gold vazio.")
    erros, vistos = [], set()
    for n, it in enumerate(itens, start=1):
        rotulo = it.get("id") or f"item {n}"
        faltando = [c for c in CAMPOS_GOLD if c not in it]
        if faltando:
            erros.append(f"{rotulo}: campo(s) ausente(s): {', '.join(faltando)}")
            continue
        if it["tipo"] not in TIPOS:
            erros.append(f"{rotulo}: tipo '{it['tipo']}' inválido; use {', '.join(TIPOS)}")
        if not isinstance(it["evidencia"], list):
            erros.append(f"{rotulo}: 'evidencia' deve ser uma lista")
        if it["id"] in vistos:
            erros.append(f"{rotulo}: id repetido")
        vistos.add(it["id"])
    if erros:
        raise ErroUsuario(f"{origem}: gold inválido:\n  - " + "\n  - ".join(erros[:20]))


In [ ]:
%%writefile /content/qa_manual_colab/configs/base.yaml
# Parâmetros comuns a todas as configurações. S0.yaml a S3.yaml sobrescrevem apenas `retriever`.
# Caminhos relativos são resolvidos a partir do diretório em que o comando é executado.
paths:
  manual_pdf: data/manual.pdf
  trechos: data/trechos.jsonl
  index_dir: index
  runs_dir: runs
  gold_dev: data/gold_dev.jsonl
  gold_test: data/gold_test.jsonl
  ppl_holdout: data/ppl_holdout.txt

ollama:
  host: http://localhost:11434          # só endereços locais são aceitos
  timeout_s: 300

modelos:
  leitor_base: qwen3:4b
  leitor_ajustado: qwen3-manual:4b
  embeddings: bge-m3
  gerador: llama3.2:3b

ingestao:
  remover_cabecalhos_rodapes: true
  limiar_repeticao_cabecalho: 0.5     # linha presente em >50% das páginas é cabeçalho/rodapé
  regex_titulo: '^(\d+(\.\d+){0,3})\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][^\n]{2,80}$'
  validar_sequencia_titulos: true     # numeração fora de sequência (passos "1 Faça...", linhas de tabela) vira texto
  min_chars_paragrafo: 25

trechos:
  max_tokens: 400
  min_tokens: 60                      # trechos menores são fundidos com o seguinte do mesmo tema
  tokens_por_palavra: 1.3             # aproximação para português
  sobreposicao_paragrafos: 0          # parágrafos repetidos entre trechos consecutivos (0 = sem sobreposição)

bm25:
  k1: 1.5
  b: 0.75
  remover_stopwords: true
  manter_acentos: true

denso:
  dimensao: 1024
  normalizar_l2: true
  batch_size: 16

retriever: hibrido                    # nenhum | bm25 | denso | hibrido
k: 5
k_candidatos_por_lista: 20            # cada lista traz 20 antes da fusão
rrf_c: 60

decodificacao:
  temperature: 0
  top_p: 1.0
  seed: 42
  num_ctx: 8192
  num_predict: 200
  think: false

abstencao:
  frase: "Não encontrado no manual"
  limiar_score: null                  # null = desligado; se número, abstém sem chamar o LLM quando o melhor score < limiar

gold:
  n_candidatas: 150
  frac_sem_resposta: 0.20
  frac_dev: 0.30
  seed: 42

avaliacao:
  configs: [S0, S1, S2, S3]
  leitores: [base, ajustado]
  repeticoes: 1
  bertscore_modelo: neuralmind/bert-base-portuguese-cased
  bertscore_num_layers: 9
  ppl_ctx: 2048


In [ ]:
%%writefile /content/qa_manual_colab/prompts/leitor.txt
Você é um assistente que responde perguntas sobre um manual técnico do setor portuário.
Use APENAS os trechos abaixo. Se a informação não estiver neles, responda exatamente:
Não encontrado no manual.
Responda em uma ou duas frases, em português, e termine indicando a fonte no formato:
(seção {{tema_id}}, p. {{pagina}})

Trechos:
{trechos}

Pergunta: {pergunta}
Resposta:


In [ ]:
%%writefile /content/qa_manual_colab/prompts/leitor_s0.txt
Você é um assistente que responde perguntas sobre um manual técnico do setor portuário.
Responda em uma ou duas frases, em português. Se não souber, responda exatamente:
Não encontrado no manual.

Pergunta: {pergunta}
Resposta:


In [ ]:
%%writefile /content/qa_manual_colab/prompts/gerador_pares_treino.txt
A seguir está um trecho de um documento do setor portuário.
Escreva UMA pergunta em português que possa ser respondida apenas com esse trecho,
do tipo "{tipo}" (factual = fato pontual, prazo, nome, quantidade; procedimental = como fazer algo, em passos).
Dê também a resposta correta em uma ou duas frases, copiando ou parafraseando o trecho,
terminando com a fonte exatamente no formato (seção {tema_id}, p. {pagina}).
Responda somente com JSON válido no formato {{"pergunta": "...", "resposta": "..."}}.

Trecho (seção {tema_id}, p. {pagina}): {texto}


In [ ]:
%%writefile /content/qa_manual_colab/prompts/gerador_sem_resposta.txt
A seguir está um trecho de um manual técnico do setor portuário.
Escreva UMA pergunta em português, plausível para quem consulta esse manual, sobre um detalhe
que NÃO aparece no trecho (por exemplo, um prazo, responsável, valor ou documento que o trecho não menciona).
Responda somente com JSON válido no formato {{"pergunta": "..."}}.

Trecho: {texto}


In [ ]:
sys.path.insert(0, PASTA_CODIGO)
import nltk

import qa_manual

assert qa_manual.__file__.startswith(PASTA_CODIGO), qa_manual.__file__
nltk.download("stopwords", quiet=True)
print("qa_manual", qa_manual.__version__, "carregado de", PASTA_CODIGO)

## 4. Documento público → trechos (passos 1 a 3)

Envie o PDF do regulamento portuário público. O mesmo código e os mesmos parâmetros de `configs/base.yaml` do
projeto são usados, para que os trechos tenham o formato que o leitor verá em produção.

In [ ]:
from pathlib import Path

from google.colab import files

DOCUMENTO_E_PUBLICO = False  # mude para True só depois de confirmar que o PDF é um documento público
assert DOCUMENTO_E_PUBLICO, "Confirme que o PDF é público (nunca o manual da empresa) e rode a célula de novo."

enviado = files.upload()
assert len(enviado) == 1, "envie exatamente um PDF"
PDF_PUBLICO = Path(next(iter(enviado)))
proibidos = {"manual.pdf", "trechos.jsonl", "trechos_stats.json"}
assert PDF_PUBLICO.name.lower() not in proibidos and "data/" not in str(PDF_PUBLICO), (
    "Este nome é o do manual confidencial; o Colab só pode receber o documento público."
)
assert PDF_PUBLICO.suffix.lower() == ".pdf", "o arquivo precisa ser um PDF"
print(PDF_PUBLICO, PDF_PUBLICO.stat().st_size, "bytes")

In [ ]:
from qa_manual import chunking, config, ingest
from qa_manual.io_utils import escrever_jsonl

cfg = config.carregar()  # configs/base.yaml embutido na seção 3
info = {}
paragrafos = ingest.construir_paragrafos(PDF_PUBLICO, cfg, info)
trechos = chunking.agrupar_trechos(paragrafos, cfg)
stats = chunking.estatisticas(trechos, cfg, info)
escrever_jsonl("trechos_publico.jsonl", [t.como_dict() for t in trechos])
print({k: stats[k] for k in ("n_paginas", "n_titulos", "n_trechos", "n_temas", "tokens")})

## 5. Passo 5 — pares de treino

Para cada trecho (até `N_MAX_TRECHOS`), o `llama3.2:3b` gera uma pergunta e a resposta com
`prompts/gerador_pares_treino.txt`. A resposta sempre termina com a fonte do próprio trecho, `(seção X, p. N)`,
preenchida pelo código. Em 20% dos trechos a pergunta é gerada com `prompts/gerador_sem_resposta.txt` e o alvo é
exatamente `Não encontrado no manual.`. O prompt de cada exemplo é montado por `qa_manual.generate.montar_prompt`,
o mesmo usado na inferência, com o trecho-evidência e 2 a 4 distratores recuperados por BM25 (ou sorteados), em
ordem aleatória.

In [ ]:
import json
import random

import bm25s
from tqdm.auto import tqdm

from qa_manual import gold, prompts
from qa_manual.generate import montar_prompt, rotulo_secao
from qa_manual.ollama_client import OllamaReal
from qa_manual.tokenize_pt import tokenizar_cfg
from qa_manual.verify import remover_citacoes

rng = random.Random(SEED)
client = OllamaReal("http://localhost:11434", timeout_s=300)
GERADOR = cfg.modelos.gerador
FRASE = cfg.abstencao.frase + "."

bm25 = bm25s.BM25(k1=cfg.bm25.k1, b=cfg.bm25.b)
bm25.index([tokenizar_cfg(t.texto, cfg) or ["_vazio_"] for t in trechos], show_progress=False)


def distratores(pergunta, evidencia, n):
    toks = tokenizar_cfg(pergunta, cfg)
    candidatos = []
    if toks:
        docs, _ = bm25.retrieve([toks], k=min(len(trechos), n + 3), show_progress=False)
        candidatos = [trechos[int(i)] for i in docs[0] if trechos[int(i)].id != evidencia.id]
    resto = [t for t in trechos if t.id != evidencia.id and t not in candidatos]
    rng.shuffle(resto)
    escolhidos = []
    usados = evidencia.n_tokens
    for t in candidatos + resto:
        if len(escolhidos) == n:
            break
        if usados + t.n_tokens <= MAX_TOKENS_CONTEXTO:
            escolhidos.append(t)
            usados += t.n_tokens
    return escolhidos


elegiveis = [t for t in trechos if t.n_tokens >= 40]
amostra = rng.sample(elegiveis, min(N_MAX_TRECHOS, len(elegiveis)))
n_sem = round(len(amostra) * FRAC_SEM_RESPOSTA)
contagem, exemplos = {}, []
for i, t in enumerate(tqdm(amostra, desc="pares")):
    citacao = f"(seção {rotulo_secao(t)}, p. {t.pagina_inicio})"
    if i < n_sem:
        p = prompts.preencher(prompts.carregar("gerador_sem_resposta"), texto=t.texto)
        dados = gold.gerar_json(client, GERADOR, p, ("pergunta",), cfg, contagem)
        alvo = FRASE
    else:
        tipo = "procedimental" if i % 5 in (1, 3) else "factual"
        p = prompts.preencher(
            prompts.carregar("gerador_pares_treino"),
            tipo=tipo,
            tema_id=rotulo_secao(t),
            pagina=t.pagina_inicio,
            texto=t.texto,
        )
        dados = gold.gerar_json(client, GERADOR, p, ("pergunta", "resposta"), cfg, contagem)
        if dados:
            alvo = f"{remover_citacoes(dados['resposta']).rstrip(' .')} {citacao}"
    if not dados:
        continue
    contexto = [t] + distratores(dados["pergunta"], t, rng.randint(2, 4))
    rng.shuffle(contexto)
    exemplos.append({"prompt": montar_prompt(dados["pergunta"], contexto), "resposta": alvo})

rng.shuffle(exemplos)
n_val = max(1, round(len(exemplos) * FRAC_VALIDACAO))
escrever_jsonl("validacao.jsonl", exemplos[:n_val])
escrever_jsonl("treino.jsonl", exemplos[n_val:])
print(
    f"{len(exemplos)} exemplos ({n_val} de validação); JSON válido em "
    f"{contagem.get('validas', 0)}/{contagem.get('tentativas', 0)} tentativas"
)
print(exemplos[0]["prompt"][-400:], "\n→", exemplos[0]["resposta"])

In [ ]:
# Libera a GPU: o Ollama não é mais necessário.
servidor_ollama.terminate()
!pkill -f "ollama serve" || true

## 6. Passo 6 — QLoRA com Unsloth

`unsloth/Qwen3-4B` em 4 bits, LoRA `r=16`, `alpha=16`, `dropout=0` nas projeções de atenção e MLP. Chat template
do Qwen3 com `enable_thinking=False`; a perda é calculada só nos tokens da resposta.

In [ ]:
from unsloth import FastLanguageModel, is_bf16_supported  # importar antes de transformers/trl

MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "treino.jsonl", "validation": "validacao.jsonl"})


def formatar(ex):
    msgs = [{"role": "user", "content": ex["prompt"]}, {"role": "assistant", "content": ex["resposta"]}]
    texto = tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
    return {"text": texto, "n_tokens": len(tokenizer(texto).input_ids)}


ds = ds.map(formatar, remove_columns=["prompt", "resposta"])
antes = {k: len(v) for k, v in ds.items()}
ds = ds.filter(lambda ex: ex["n_tokens"] <= MAX_SEQ).remove_columns("n_tokens")  # truncado perderia a resposta
print("exemplos (antes → depois do filtro de tamanho):", antes, {k: len(v) for k, v in ds.items()})

# Marcadores do turno, extraídos do próprio template (inclui o bloco <think> vazio do Qwen3).
amostra = tokenizer.apply_chat_template(
    [{"role": "user", "content": "PERGUNTA"}, {"role": "assistant", "content": "RESPOSTA"}],
    tokenize=False,
    enable_thinking=False,
)
PARTE_USUARIO = "<|im_start|>user\n"
PARTE_RESPOSTA = amostra[amostra.rindex("<|im_start|>assistant") : amostra.rindex("RESPOSTA")]
print(repr(PARTE_RESPOSTA))

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=2,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not is_bf16_supported(),  # T4: fp16=True, bf16=False
        bf16=is_bf16_supported(),
        seed=SEED,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        output_dir="saidas",
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part=PARTE_USUARIO, response_part=PARTE_RESPOSTA)

# Conferência da máscara: só a resposta deve aparecer nos rótulos.
rotulos = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
print("rótulos do 1º exemplo:", repr(tokenizer.decode(rotulos)))

In [ ]:
import pandas as pd

resultado = trainer.train()
log = pd.DataFrame(trainer.state.log_history)
colunas = [c for c in ("step", "epoch", "loss", "eval_loss", "learning_rate", "grad_norm") if c in log.columns]
log[colunas].to_csv("treino_log.csv", index=False)
print(log[colunas].dropna(how="all", subset=[c for c in ("loss", "eval_loss") if c in colunas]).tail(10))
model.save_pretrained("adapter")
tokenizer.save_pretrained("adapter")

## 7. Checagem de sanidade

Dez exemplos da validação respondidos pelo modelo ajustado (via `transformers`): a resposta deve trazer a fonte
no formato `(seção X, p. N)` e, nos exemplos sem resposta, usar a frase de abstenção.

In [ ]:
from qa_manual.verify import detectar_abstencao, extrair_citacoes

FastLanguageModel.for_inference(model)
validacao = [json.loads(l) for l in open("validacao.jsonl", encoding="utf-8")][:10]
checagens = []
for ex in validacao:
    entrada = tokenizer.apply_chat_template(
        [{"role": "user", "content": ex["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    ids = tokenizer(entrada, return_tensors="pt").to("cuda")
    saida = model.generate(**ids, max_new_tokens=160, do_sample=False)
    resposta = tokenizer.decode(saida[0][ids["input_ids"].shape[1] :], skip_special_tokens=True).strip()
    sem_resposta = ex["resposta"] == FRASE
    ok = detectar_abstencao(resposta, cfg.abstencao.frase) if sem_resposta else bool(extrair_citacoes(resposta))
    checagens.append({"sem_resposta": sem_resposta, "formato_ok": ok, "resposta": resposta, "alvo": ex["resposta"]})
    print(("OK " if ok else "!! ") + resposta[:160])

metricas = {
    "eval_loss_final": next((h["eval_loss"] for h in reversed(trainer.state.log_history) if "eval_loss" in h), None),
    "train_loss": resultado.training_loss,
    "formato_ok": sum(c["formato_ok"] for c in checagens) / len(checagens),
    "n_treino": len(ds["train"]),
    "n_validacao": len(ds["validation"]),
}
json.dump(metricas, open("metricas_validacao.json", "w"), ensure_ascii=False, indent=2)
metricas

## 8. Passo 7 — exportação em GGUF `q4_k_m`

O Unsloth faz o merge do adaptador e a conversão com o llama.cpp (pode levar alguns minutos).

In [ ]:
import glob
import shutil

model.save_pretrained_gguf("qwen3-manual", tokenizer, quantization_method="q4_k_m")
gerados = [g for g in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in g.lower()]
assert gerados, "nenhum GGUF q4_k_m encontrado"
shutil.copy(gerados[0], "qwen3-manual-q4_k_m.gguf")
!ls -lh qwen3-manual-q4_k_m.gguf

In [ ]:
for nome in ("qwen3-manual-q4_k_m.gguf", "treino_log.csv", "metricas_validacao.json"):
    files.download(nome)

## 9. Na máquina local

```bash
mkdir -p models && mv ~/Downloads/qwen3-manual-q4_k_m.gguf models/
mv ~/Downloads/treino_log.csv ~/Downloads/metricas_validacao.json colab/
bash colab/criar_modelo_local.sh models/qwen3-manual-q4_k_m.gguf   # Modelfile = o do qwen3:4b com outro FROM
qa-manual smoke --sem-bertscore
```

O script gera o `Modelfile` a partir de `ollama show qwen3:4b --modelfile`, trocando só a linha `FROM`, cria
`qwen3-manual:4b` e faz um teste com `ollama run`.